# yolov5 cuDNN backend — parity + speed (foolproof, one-click)
Builds `dnet_test` with the **cuDNN conv backend** (`-DUSE_CUDA -DUSE_CUDNN -lcudnn`) and checks
it reproduces the trusted CPU engine, then times cuDNN-vs-cuBLAS training on COCO128.
The cuDNN header/lib path is auto-detected (works with Colab's torch-bundled cuDNN).
**Runtime → GPU (T4)**, then Run all.


In [ ]:
!nvidia-smi -L
!nvcc --version | tail -1


In [ ]:
%cd /content
!rm -rf yolov5_cpp
!git clone -q --branch main https://github.com/yomei-o/yolov5_cpp.git
%cd /content/yolov5_cpp


### Locate cuDNN (header + lib) — auto


In [ ]:
import os, glob
inc = lib = None
# 1) pip nvidia-cudnn (ships with torch on Colab)
try:
    import nvidia.cudnn
    d = os.path.dirname(nvidia.cudnn.__file__)
    if os.path.exists(d + "/include/cudnn.h"):
        inc, lib = d + "/include", d + "/lib"
except Exception:
    pass
# 2) system locations
if not inc:
    for h in ["/usr/include/cudnn.h"] + glob.glob("/usr/include/**/cudnn.h", recursive=True) + glob.glob("/usr/local/cuda*/include/cudnn.h"):
        if os.path.exists(h):
            inc, lib = os.path.dirname(h), "/usr/lib/x86_64-linux-gnu"; break
assert inc, "cudnn.h not found -> run: !pip install nvidia-cudnn-cu12  then re-run this cell"
os.environ["CUDNN_INC"], os.environ["CUDNN_LIB"] = inc, lib
print("CUDNN_INC =", inc)
print("CUDNN_LIB =", lib)


### 1. Parity — baseline CUDA (sanity, no cuDNN)


In [ ]:
!nvcc -x cu -O2 -std=c++17 --extended-lambda -arch=native -DUSE_CUDA -Ipure/third_party pure/dnet5_test.cpp -o dnet5_cuda
!./dnet5_cuda; echo "exit=$?"


### 2. Parity — cuDNN conv backend  (the new path)


In [ ]:
!nvcc -x cu -O2 -std=c++17 --extended-lambda -arch=native -DUSE_CUDA -DUSE_CUDNN \
      -I"$CUDNN_INC" -L"$CUDNN_LIB" -Ipure/third_party pure/dnet5_test.cpp -lcudnn -o dnet5_cudnn
!LD_LIBRARY_PATH="$CUDNN_LIB:$LD_LIBRARY_PATH" ./dnet5_cudnn; echo "exit=$?"


**Expected:** both print `worst |device - CPU-engine| = ... MATCH`. cuDNN may show a larger diff (~1e-2) if it picks a TF32 algo on Ampere+ — that's fine.


### 3. Speed — cuDNN vs cuBLAS training (COCO128, 3 epochs)


In [ ]:
!wget -q https://github.com/ultralytics/yolov5/releases/download/v1.0/coco128.zip && unzip -q -o coco128.zip


In [ ]:
%%time
# cuBLAS conv (im2col + cuBLAS GEMM)
!nvcc -x cu -O2 -std=c++17 --extended-lambda -arch=native -DUSE_CUDA -DUSE_CUBLAS -Ipure/third_party pure/dtrain_coco5.cpp -lcublas -o dtrain_cublas
!./dtrain_cublas coco128/images/train2017 320 4 3


In [ ]:
%%time
# cuDNN conv (+cuBLAS for the non-conv gemms)
!nvcc -x cu -O2 -std=c++17 --extended-lambda -arch=native -DUSE_CUDA -DUSE_CUDNN -DUSE_CUBLAS \
      -I"$CUDNN_INC" -L"$CUDNN_LIB" -Ipure/third_party pure/dtrain_coco5.cpp -lcudnn -lcublas -o dtrain_cudnn
!LD_LIBRARY_PATH="$CUDNN_LIB:$LD_LIBRARY_PATH" ./dtrain_cudnn coco128/images/train2017 320 4 3


Compare the **s/epoch** the two runs print. cuDNN should be faster on the conv-heavy yolov5n at 640; the loss curves should track each other (same math).
